# ⚡ fade_movers — the day's steepest 1-minute moves, to look at

**Research only: no strategy, no backtest.** This notebook finds, for every stock with continuous
1-minute data, the steepest rise and the steepest fall of each day — the biggest change over any
`WINDOW` minutes (15 by default) — and draws the 1-minute candles of the week's biggest ones.

1. **The scan** reads the 1-minute bars week by week and keeps one row per stock-day.
2. **The screen** keeps the stocks with 1-minute data on (nearly) every session of the window.
3. **The weekly list** ranks the steepest rises and falls of each week.
4. **The candles** draw those days, with the steep window shaded.

The first run reads ~45 weeks from the database (a few minutes); every week is cached in
`var/fade_movers/`, so later runs and an interrupted run continue where they stopped.

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel (README "Quick start")
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import text

import trading_simple
from trading_simple import plots
from trading_simple.db import make_engine

ENGINE = make_engine()
REPO = Path(trading_simple.__file__).resolve().parents[2]
CACHE = REPO / "var" / "fade_movers"
TZ = "America/New_York"

# --- knobs -------------------------------------------------------------------------------
START, END = "2025-06-18", "2026-04-30"  # 1m history starts 2025-06-18. END stops before the
                                        # locked holdout (2026-05-01 on): reading it spends it.
WINDOW = 15            # minutes: a "steep" move is the biggest change over this many minutes
COVERAGE = 0.95        # continuous = a full day of 1m bars on at least this share of sessions
MIN_PRICE = 1.0        # previous close at least this ($)
MIN_DAY_DOLLARS = 10e6 # previous day's 1m dollar volume at least this ($)
TOP_PER_WEEK = 3       # steepest rises and falls listed per week
# Half-days: the stored "regular" session runs past the 13:00 close, so they are skipped.
HALF_DAYS = {"2025-07-03", "2025-11-28", "2025-12-24", "2026-11-27", "2026-12-24"}

## 1 · The steep-change method

For one stock on one day, take its prices in time order: the day's first trade (the open of the
first minute that traded), then every minute's close. Minutes with no trade are skipped: IBKR stores
a minute with fewer than 100 shares as a copy of the previous close with volume 0, so those minutes
carry the last real price forward instead of a fake one.

The **change over `WINDOW` minutes** ending at a minute is `price now / price WINDOW minutes earlier − 1`.
The day's **steepest rise** is the largest of these changes, its **steepest fall** the most negative.
Every window lies inside the session: the overnight gap never counts as a steep move (a gap at
the open counts only from the first trade on), so a split or an earnings gap does not show up
here unless the stock keeps running after the open.

Each stock-day row also keeps the window's start and end time, the dollars traded in the window,
the day's first trade, last trade and dollar volume. The day's return (last trade against the
previous session's last trade) is added after the scan.

In [ ]:
FIELDS = ("open", "high", "low", "close", "volume")


def mondays(start, end):
    """The Monday of every week from START to END."""
    first = pd.Timestamp(start) - pd.Timedelta(days=pd.Timestamp(start).weekday())
    return list(pd.date_range(first, end, freq="7D"))


def load_week(monday):
    """Every symbol's regular-session 1m bars of one week, as wide frames (UTC minute x symbol)."""
    a = pd.Timestamp(monday, tz="UTC")
    b = min(a + pd.Timedelta(days=5), pd.Timestamp(END, tz="UTC") + pd.Timedelta(days=1))
    sql = text("SELECT symbol, ts, open, high, low, close, volume FROM prices_ohlcv "
               "WHERE bar = '1m' AND rth AND ts >= :a AND ts < :b")
    with ENGINE.connect() as conn:
        long = pd.read_sql(sql, conn, params={"a": a, "b": b})
    if long.empty:
        return None
    long["ts"] = pd.to_datetime(long["ts"], utc=True)
    return {f: long.pivot(index="ts", columns="symbol", values=f) for f in FIELDS}


def steep_day(wide, day):
    """One row per symbol traded on ``day``: its steepest WINDOW-minute rise and fall."""
    grid = pd.date_range(pd.Timestamp(f"{day} 09:30", tz=TZ), periods=390, freq="min")
    grid = grid.tz_convert("UTC")
    f = {k: wide[k].reindex(grid) for k in FIELDS}
    rows = f["volume"].notna().sum()
    keep = rows[rows > 0].index
    f = {k: v[keep] for k, v in f.items()}
    traded = (f["volume"] > 0) & f["close"].notna()          # volume 0 = placeholder, no trade
    o, c = f["open"].where(traded), f["close"].where(traded)
    dollars = (c * f["volume"]).fillna(0.0).to_numpy()
    first = o.bfill().iloc[0].to_numpy()                      # the day's first trade
    path = np.vstack([first, c.ffill().to_numpy()])           # 391 prices: first trade, closes
    change = path[WINDOW:] / path[:-WINDOW] - 1               # row i: window = minutes i .. i+W-1
    cum = np.vstack([np.zeros(len(keep)), dollars.cumsum(0)])
    window_dollars = cum[WINDOW:] - cum[:-WINDOW]
    cols = np.arange(len(keep))
    up = np.where(np.isfinite(change), change, -np.inf).argmax(0)
    dn = np.where(np.isfinite(change), change, np.inf).argmin(0)
    start = lambda i: grid[i]                                # minute i's open time  # noqa: E731
    end = lambda i: grid[i + WINDOW - 1] + pd.Timedelta(minutes=1)  # noqa: E731 (last close)
    return pd.DataFrame({
        "date": pd.Timestamp(day), "symbol": keep, "rows": rows[keep].to_numpy(),
        "traded": traded.sum().to_numpy(), "first": first, "last": path[-1],
        "day_dollars": dollars.sum(0),
        "up": change[up, cols], "up_start": start(up), "up_end": end(up),
        "up_dollars": window_dollars[up, cols],
        "down": change[dn, cols], "down_start": start(dn), "down_end": end(dn),
        "down_dollars": window_dollars[dn, cols],
    })


def scan_week(monday):
    """All stock-days of one week (cached in var/fade_movers/)."""
    path = CACHE / f"steep_w{WINDOW}_{monday:%Y-%m-%d}.pkl"
    if path.exists():
        return pd.read_pickle(path)
    wide = load_week(monday)
    days = []
    if wide is not None:
        local = wide["close"].index.tz_convert(TZ)
        for day in sorted(set(local.strftime("%Y-%m-%d"))):
            if START <= day <= END and day not in HALF_DAYS:
                days.append(steep_day(wide, day))
    out = pd.concat(days, ignore_index=True) if days else pd.DataFrame()
    CACHE.mkdir(parents=True, exist_ok=True)
    if monday + pd.Timedelta(days=7) <= pd.Timestamp(END):   # never cache a cut-off week
        out.to_pickle(path)
    return out

In [ ]:
weeks = mondays(START, END)
parts = []
for k, monday in enumerate(weeks, 1):
    parts.append(scan_week(monday))
    print(f"\r{k}/{len(weeks)} weeks  ({monday:%Y-%m-%d})", end="")
days = pd.concat(parts, ignore_index=True).sort_values(["symbol", "date"], ignore_index=True)

# previous session's last trade and dollar volume, only when that session is the day before
sessions = pd.Index(sorted(days["date"].unique()))
pos = sessions.get_indexer(days["date"])
same = days["symbol"].eq(days["symbol"].shift()) & (pos - np.roll(pos, 1) == 1)
days["prev_last"] = days["last"].shift().where(same)
days["prev_dollars"] = days["day_dollars"].shift().where(same)
days["day_ret"] = days["last"] / days["prev_last"] - 1
print(f"\n{len(days):,} stock-days, {days['symbol'].nunique():,} symbols, {len(sessions)} sessions "
      f"({sessions[0]:%Y-%m-%d} → {sessions[-1]:%Y-%m-%d})")

## 2 · The screen: stocks with continuous 1-minute data

A session counts as **full** for a stock when at least 380 of its 390 regular minutes are stored
(a stored minute may be a no-trade placeholder; that is IBKR's way of saying "nothing traded",
not a hole). A stock is **continuous** when it has a full day on at least `COVERAGE` of the
window's sessions.

The 1-minute pool grew from about 510 to about 1,750 names on 2026-03-25, so with the default
window only the original ~510 can be continuous. Set `START = "2026-03-25"` to screen the wider
pool over its shorter history.

In [ ]:
full = days[days["rows"] >= 380].groupby("symbol")["date"].nunique()
coverage = (full / len(sessions)).sort_values(ascending=False)
continuous = sorted(coverage[coverage >= COVERAGE].index)
print(f"{len(continuous)} of {days['symbol'].nunique()} symbols are continuous "
      f"(full day on >= {COVERAGE:.0%} of {len(sessions)} sessions)")
print("coverage of the others:", coverage[coverage < COVERAGE].describe().round(2).to_dict())
print(", ".join(continuous))

## 3 · The steepest rises and falls of each week

Among the continuous stocks priced at least `MIN_PRICE` with at least `MIN_DAY_DOLLARS` traded the
day before, each week lists the `TOP_PER_WEEK` steepest rises and falls (one day per stock per
week: its steepest). `move` is the change over the `WINDOW`-minute window, `day` the stock's
whole-day return, `$ in window` the dollars traded inside the window. Times are New York.

In [ ]:
pool = days[days["symbol"].isin(continuous) & (days["prev_last"] >= MIN_PRICE)
            & (days["prev_dollars"] >= MIN_DAY_DOLLARS)].copy()
pool["week"] = pool["date"] - pd.to_timedelta(pool["date"].dt.weekday, unit="D")


def weekly(side):
    """Each week's TOP_PER_WEEK steepest moves on one side ('up' or 'down')."""
    s = pool.sort_values(side, ascending=side == "down")
    s = s.drop_duplicates(["week", "symbol"]).groupby("week").head(TOP_PER_WEEK)
    t = lambda c: s[c].dt.tz_convert(TZ).dt.strftime("%H:%M")  # noqa: E731
    return pd.DataFrame({
        "week": s["week"].dt.strftime("%Y-%m-%d"), "side": side, "symbol": s["symbol"],
        "date": s["date"].dt.strftime("%Y-%m-%d"),
        "window": t(f"{side}_start") + "–" + t(f"{side}_end"),
        "move %": (100 * s[side]).round(1), "day %": (100 * s["day_ret"]).round(1),
        "$ in window (M)": (s[f"{side}_dollars"] / 1e6).round(1),
        "prev close": s["prev_last"].round(2)}).sort_values(["week", "move %"],
                                                            ascending=[True, side == "down"])


movers = pd.concat([weekly("up"), weekly("down")], ignore_index=True)
med = movers.groupby("side")["move %"].median()
print(f"{movers['week'].nunique()} weeks; median listed {WINDOW}-min rise {med['up']:+.1f}%, "
      f"fall {med['down']:+.1f}%")
movers.sort_values(["week", "side", "move %"], ascending=[False, False, False]).head(60)

## 3b · When in the day do the steep moves happen?

Each stock-day's steepest window is placed by its **start** time. If steep moves were spread
evenly over the session, each bucket would hold its share of the possible start minutes (first
row, "evenly spread"). A window that starts at 09:30 begins at the day's first trade, so the
overnight gap is not part of the move: what is counted is how far the price runs in the first
`WINDOW` minutes of trading. Rows are % of the stock-days in that row; same pool as the table above.

In [ ]:
EDGES = [0, 15, 30, 60, 120, 240, 330, 391 - WINDOW]   # minutes after 09:30 (window start)
LABELS = ["09:30-09:45", "09:45-10:00", "10:00-10:30", "10:30-11:30", "11:30-13:30",
          "13:30-15:00", "15:00 on"]


def when(frame, side):
    """Share (%) of ``frame``'s steepest ``side`` windows starting in each time bucket."""
    t = frame[f"{side}_start"].dt.tz_convert(TZ)
    minute = t.dt.hour * 60 + t.dt.minute - 570               # minutes after 09:30
    bucket = pd.cut(minute, EDGES, right=False, labels=LABELS)
    return (100 * bucket.value_counts(normalize=True)).reindex(LABELS)


table = {("evenly spread", ""): pd.Series(100 * np.diff(EDGES) / EDGES[-1], index=LABELS)}
for side in ("up", "down"):
    for th in (0.0, 0.03, 0.05, 0.10):
        sub = pool[pool[side].abs() >= th]
        table[(side, f"|move| >= {th:.0%} (n={len(sub):,})")] = when(sub, side)
    top = (pool.sort_values(side, ascending=side == "down")
           .drop_duplicates(["week", "symbol"]).groupby("week").head(TOP_PER_WEEK))
    table[(side, f"weekly top {TOP_PER_WEEK} (n={len(top)})")] = when(top, side)
pd.DataFrame(table).T.round(0)

## 4 · The candles

`plot_day` draws one stock-day's 1-minute candles, with the session before and the session
after beside it (nights, weekends and holidays squeezed out) and volume underneath, the steep window shaded
(green for a rise, red for a fall) and the previous close as a dotted line. Minutes without a
trade (the volume-0 placeholders) are left out, so a gap between candles means nothing traded.

The next cell draws every mover of one week (`WEEK`, default the latest); the cell after it draws
any stock and day you name.

In [ ]:
def plot_day(symbol, date, side=None):
    """1m candles of one stock-day plus the sessions before and after it.

    The day's steepest window is shaded: ``side`` 'up' or 'down', or None for both.
    """
    date = pd.Timestamp(date)
    row = days[(days["symbol"] == symbol) & (days["date"] == date)].iloc[0]
    i = sessions.get_loc(date)
    shown = sessions[max(i - 1, 0): i + 2]                    # the session before, the day, after
    a = pd.Timestamp(f"{shown[0]:%Y-%m-%d} 09:30", tz=TZ).tz_convert("UTC")
    b = pd.Timestamp(f"{shown[-1]:%Y-%m-%d} 16:00", tz=TZ).tz_convert("UTC")
    sql = text("SELECT ts, open, high, low, close, volume FROM prices_ohlcv WHERE symbol = :s "
               "AND bar = '1m' AND rth AND ts >= :a AND ts < :b ORDER BY ts")
    with ENGINE.connect() as conn:
        bars = pd.read_sql(sql, conn, params={"s": symbol, "a": a, "b": b})
    bars = bars[bars["volume"] > 0]                          # placeholders: no trade, no candle
    x = pd.to_datetime(bars["ts"], utc=True).dt.tz_convert(TZ).dt.tz_localize(None)
    keep = x.dt.normalize().isin(shown) & (x.dt.hour < 16)  # the shown sessions only
    bars, x = bars[keep], x[keep]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.78, 0.22],
                        vertical_spacing=0.03)
    fig.add_trace(go.Candlestick(
        x=x, open=bars["open"], high=bars["high"], low=bars["low"], close=bars["close"],
        increasing={"line": {"color": plots._WIN}, "fillcolor": plots._WIN},
        decreasing={"line": {"color": plots._LOSS}, "fillcolor": plots._LOSS},
        name=symbol, showlegend=False), row=1, col=1)
    fig.add_trace(go.Bar(x=x, y=bars["volume"], marker_color=plots._MUTED, name="volume",
                         showlegend=False), row=2, col=1)
    # shapes after the traces: a shape aimed at an empty subplot is dropped silently
    if np.isfinite(row["prev_last"]):
        fig.add_hline(y=row["prev_last"], line={"dash": "dot", "color": plots._MUTED},
                      annotation_text="close before the day", row=1, col=1)
    sides = [side] if side else ["up", "down"]
    for s in sides:
        t0, t1 = (row[f"{s}_{k}"].tz_convert(TZ).tz_localize(None) for k in ("start", "end"))
        fig.add_vrect(x0=t0, x1=t1, fillcolor=plots._WIN if s == "up" else plots._LOSS,
                      opacity=0.15, line_width=0, layer="below", row=1, col=1)
    rets = days[(days["symbol"] == symbol) & days["date"].isin(shown)].set_index("date")["day_ret"]
    for d in shown:
        if d != shown[0]:                                    # a dotted line at each new open
            fig.add_vline(x=pd.Timestamp(f"{d:%Y-%m-%d} 09:30"),
                          line={"dash": "dot", "color": plots._MUTED}, row="all", col=1)
        r = rets.get(d, np.nan)
        label = f"{d:%a %Y-%m-%d}" + (f" · day {100 * r:+.1f}%" if np.isfinite(r) else "")
        fig.add_annotation(x=pd.Timestamp(f"{d:%Y-%m-%d} 12:45"), y=1.0, yref="paper",
                           yanchor="bottom", showarrow=False,
                           text=f"<b>{label}</b>" if d == date else label)
    moves = "  ".join(f"{'rise' if s == 'up' else 'fall'} {100 * row[s]:+.1f}% "
                      f"({row[f'{s}_start'].tz_convert(TZ):%H:%M}–"
                      f"{row[f'{s}_end'].tz_convert(TZ):%H:%M})" for s in sides)
    plots._base(fig, title=f"{symbol} · {date:%Y-%m-%d} · {WINDOW}-min {moves}", height=560)
    fig.update_layout(margin={"t": 80})
    # hide nights, weekends and weekday closures, so the three sessions sit side by side
    gaps = pd.bdate_range(shown[0], shown[-1]).difference(shown)
    fig.update_xaxes(rangeslider_visible=False, rangebreaks=[
        {"bounds": ["sat", "mon"]}, {"bounds": [16, 9.5], "pattern": "hour"},
        {"values": [f"{g:%Y-%m-%d}" for g in gaps]}])
    return fig

In [ ]:
WEEK = movers["week"].max()          # e.g. "2025-11-17"; any Monday in the table
for _, m in movers[movers["week"] == WEEK].iterrows():
    plot_day(m["symbol"], m["date"], m["side"]).show()

In [ ]:
# any stock-day: plot_day(symbol, "YYYY-MM-DD") shades both its steepest rise and fall
plot_day(movers.iloc[0]["symbol"], movers.iloc[0]["date"])